# 🛰️ Google Earth Engine + Jupyter Notebook + ArcGIS Pro
## Del satélite al mapa: NDVI con Sentinel-2 y exportación automática a ArcGIS Pro

**Caso de estudio:** departamento de Zacapa, Guatemala (Corredor Seco Centroamericano)

---

### ¿Qué vamos a lograr?
Al terminar este ejercicio tendrá, sin salir de ArcGIS Pro:

1. Un **compuesto Sentinel-2 sin nubes** para el período de interés.
2. Un raster de **NDVI** calculado en la nube con Google Earth Engine.
3. **Estadísticas** del NDVI para el área de estudio.
4. Un **GeoTIFF** exportado a la carpeta del proyecto y **agregado automáticamente** al mapa activo, ya simbolizado.

### Flujo de trabajo

```
Sentinel-2 ─► Earth Engine ─► filtro espacial/temporal ─► máscara de nubes ─► mediana ─► NDVI
                                                                                         │
                     ArcGIS Pro ◄── ArcPy ◄── GeoTIFF ◄── estadísticas + mapa (geemap) ◄─┘
```

### Agenda sugerida (≈ 45 min)

| Bloque | Secciones | Tiempo |
|---|---|---|
| Preparación y conexión | 0 – 2 | 5 min |
| Datos de entrada | 3 – 5 | 10 min |
| Procesamiento y NDVI | 6 – 7 | 10 min |
| Visualización e interpretación | 8 – 9 | 10 min |
| Exportación a ArcGIS Pro | 10 – 12 | 10 min |

> ⚠️ **Importante:** este notebook debe ejecutarse **dentro de ArcGIS Pro** (Insert ▸ New Notebook o abrir este `.ipynb` desde el panel Catalog), porque usa `arcpy` y `ArcGISProject("CURRENT")`. Necesita un proyecto abierto con al menos un mapa.

## 0. Antes de empezar: lista de verificación

- [ ] ArcGIS Pro 3.x con un **entorno de Python clonado** (el entorno por defecto `arcgispro-py3` no permite instalar paquetes).
- [ ] Paquetes `earthengine-api` y `geemap` instalados en ese entorno.
- [ ] Cuenta de Google Earth Engine y un **proyecto de Google Cloud registrado** para Earth Engine.
- [ ] Autenticación realizada al menos una vez (`ee.Authenticate()`), idealmente **antes** del webinar.
- [ ] Un proyecto de ArcGIS Pro abierto con un mapa activo.

La siguiente celda confirma que todo está disponible.

## 1. Importar librerías y verificar versiones

In [ ]:
import os
import ee
import geemap
import arcpy

print("✓ Librerías cargadas")
print(f"  earthengine-api : {ee.__version__}")
print(f"  geemap          : {geemap.__version__}")
print(f"  ArcGIS Pro      : {arcpy.GetInstallInfo()['Version']}")

## 2. Parámetros del ejercicio

👉 **Esta es la única celda que necesita modificar.** Cambie el proyecto de Earth Engine por el suyo y, si lo desea, el departamento, el período o la resolución.

| Parámetro | Descripción |
|---|---|
| `PROYECTO_GEE` | ID de su proyecto de Google Cloud registrado en Earth Engine |
| `PAIS`, `DEPTO` | Nombres tal como aparecen en FAO GAUL (nivel 1) |
| `FECHA_INICIO`, `FECHA_FIN` | Período de análisis. **`FECHA_FIN` no se incluye** (fecha exclusiva) |
| `MAX_NUBES` | % máximo de nubosidad por escena |
| `ESCALA_M` | Resolución de exportación en metros (Sentinel-2 permite 10 m) |
| `CRS_SALIDA` | Sistema de referencia del GeoTIFF (EPSG:32616 = WGS 84 / UTM 16N, adecuado para Guatemala) |

In [ ]:
# --- Conexión ---------------------------------------------------------------
PROYECTO_GEE = 'su-proyecto-gee'        # ← reemplazar por el ID de su proyecto

# --- Área de estudio --------------------------------------------------------
PAIS  = 'Guatemala'
DEPTO = 'Zacapa'

# --- Período (FECHA_FIN es exclusiva: '2026-04-01' incluye todo marzo) -------
FECHA_INICIO = '2026-01-01'
FECHA_FIN    = '2026-04-01'

# --- Calidad y salida -------------------------------------------------------
MAX_NUBES  = 20            # % de nubosidad máxima por escena
ESCALA_M   = 30            # resolución de exportación (m)
CRS_SALIDA = 'EPSG:32616'  # WGS 84 / UTM zona 16N
NODATA     = -9999         # valor para píxeles sin dato en el GeoTIFF

print(f"Área    : {DEPTO}, {PAIS}")
print(f"Período : {FECHA_INICIO} → {FECHA_FIN} (exclusiva)")
print(f"Salida  : {ESCALA_M} m, {CRS_SALIDA}")

## 3. Conectar con Google Earth Engine

Si ya se autenticó antes, la conexión es inmediata. Si no, se abrirá el navegador para autorizar el acceso.

In [ ]:
if PROYECTO_GEE == 'su-proyecto-gee':
    raise ValueError("Defina PROYECTO_GEE en la sección 2 con el ID de su proyecto de Earth Engine.")

try:
    ee.Initialize(project=PROYECTO_GEE)
    print(f"✓ Conectado a Google Earth Engine (proyecto: {PROYECTO_GEE})")
except Exception:
    print("Autenticando Google Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=PROYECTO_GEE)
    print(f"✓ Autenticación completada (proyecto: {PROYECTO_GEE})")

## 4. Área de estudio: límite del departamento

Usamos los límites administrativos **FAO GAUL 2015, nivel 1** (departamentos), disponibles directamente en el catálogo de Earth Engine.

In [ ]:
limites = ee.FeatureCollection('FAO/GAUL/2015/level1')

depto = limites.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', PAIS),
        ee.Filter.eq('ADM1_NAME', DEPTO)
    )
)

n = depto.size().getInfo()
if n == 0:
    nombres = limites.filter(ee.Filter.eq('ADM0_NAME', PAIS)).aggregate_array('ADM1_NAME').sort().getInfo()
    raise ValueError(f"No se encontró '{DEPTO}'. Nombres disponibles en {PAIS}:\n{nombres}")

geom = depto.geometry()
area_km2 = geom.area(maxError=100).divide(1e6).getInfo()

# Centro del mapa calculado a partir del polígono (se adapta si cambia DEPTO)
lon_c, lat_c = geom.centroid(maxError=100).coordinates().getInfo()

print(f"✓ {DEPTO}: {n} polígono(s), {area_km2:,.0f} km²")
print(f"  Centro: lat {lat_c:.3f}, lon {lon_c:.3f}")

## 5. Buscar imágenes Sentinel-2

Usamos **Sentinel-2 Nivel 2A (reflectancia de superficie), colección armonizada** — `COPERNICUS/S2_SR_HARMONIZED`. La versión armonizada corrige el cambio de escala que Copernicus introdujo en enero de 2022, por lo que permite comparar fechas antes y después de ese cambio.

Filtros aplicados:
- **Espacial:** escenas que intersectan el departamento.
- **Temporal:** el período definido en los parámetros.
- **Nubosidad:** menos de `MAX_NUBES` % de nubes por escena.

In [ ]:
coleccion = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(geom)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_NUBES))
)

n_img = coleccion.size().getInfo()
if n_img == 0:
    raise ValueError("No hay imágenes con esos filtros. Amplíe el período o aumente MAX_NUBES.")

fechas = (
    coleccion.aggregate_array('system:time_start')
    .map(lambda t: ee.Date(t).format('YYYY-MM-dd'))
    .distinct().sort()
    .getInfo()
)

print(f"✓ Escenas Sentinel-2 seleccionadas: {n_img}")
print(f"  Fechas de adquisición distintas: {len(fechas)} ({fechas[0]} a {fechas[-1]})")

## 6. Máscara de nubes y compuesto temporal

Aunque filtramos escenas con poca nubosidad, cada escena puede conservar algunas nubes. La banda **SCL** (*Scene Classification Layer*) de Sentinel-2 identifica cada píxel; eliminamos:

| Clase SCL | Significado |
|---|---|
| 3 | Sombra de nube |
| 8 | Nube (probabilidad media) |
| 9 | Nube (probabilidad alta) |
| 10 | Cirros |

Luego combinamos todas las escenas con la **mediana**: para cada píxel se toma el valor central de la serie, lo que reduce el efecto de nubes residuales, sombras y valores atípicos.

In [ ]:
def enmascarar_nubes(img):
    scl = img.select('SCL')
    nubes = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
    return img.updateMask(nubes.Not())

s2 = (
    coleccion
    .map(enmascarar_nubes)
    .median()
    .clip(geom)
)

print("✓ Compuesto Sentinel-2 (mediana, sin nubes) creado")

## 7. Calcular el NDVI

$$NDVI = \frac{NIR - Rojo}{NIR + Rojo}$$

En Sentinel-2:

| Banda | Región | Longitud de onda central | Resolución |
|---|---|---|---|
| **B4** | Rojo | ≈ 665 nm | 10 m |
| **B8** | Infrarrojo cercano (NIR) | ≈ 842 nm | 10 m |

La vegetación sana absorbe el rojo (clorofila) y refleja fuertemente el NIR, por eso obtiene valores de NDVI altos.

In [ ]:
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

print("✓ NDVI calculado")

## 8. Estadísticas del NDVI en el área de estudio

Antes de exportar, conviene conocer la distribución de valores. El cálculo se hace en Earth Engine y solo regresa el resumen.

In [ ]:
reductor = (
    ee.Reducer.mean()
    .combine(ee.Reducer.minMax(), sharedInputs=True)
    .combine(ee.Reducer.percentile([10, 50, 90]), sharedInputs=True)
)

stats = ndvi.reduceRegion(
    reducer=reductor,
    geometry=geom,
    scale=ESCALA_M,
    maxPixels=1e10,
    bestEffort=True
).getInfo()

print(f"NDVI — {DEPTO}, {FECHA_INICIO} a {FECHA_FIN}")
print(f"  Media     : {stats['NDVI_mean']:.3f}")
print(f"  Mínimo    : {stats['NDVI_min']:.3f}")
print(f"  P10 / P50 / P90 : {stats['NDVI_p10']:.3f} / {stats['NDVI_p50']:.3f} / {stats['NDVI_p90']:.3f}")
print(f"  Máximo    : {stats['NDVI_max']:.3f}")

## 9. Visualizar Sentinel-2 y NDVI

El centro del mapa se define con las coordenadas calculadas en la sección 4 (en lugar de `centerObject()`, que presenta problemas de compatibilidad dentro del entorno de ArcGIS Pro).

Use el control de capas (esquina superior derecha) para alternar entre el color verdadero y el NDVI.

In [ ]:
Map = geemap.Map(
    center=[lat_c, lon_c],
    zoom=9,
    toolbar_ctrl=False,
    layer_ctrl=True
)

Map.addLayer(
    s2,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Sentinel-2 color verdadero'
)

paleta_ndvi = ['#a50026', '#f46d43', '#fee08b', '#d9ef8b', '#66bd63', '#006837']
Map.addLayer(
    ndvi,
    {'min': 0, 'max': 0.8, 'palette': paleta_ndvi},
    'NDVI'
)

Map.addLayer(
    depto.style(color='000000', fillColor='00000000', width=2),
    {},
    f'Límite de {DEPTO}'
)

Map.add_colorbar(
    {'min': 0, 'max': 0.8, 'palette': paleta_ndvi},
    label='NDVI'
)

Map

### 🔎 Cómo leer el mapa

| NDVI | Lectura general |
|---|---|
| < 0 | Agua, nubes residuales |
| 0 – 0.2 | Suelo desnudo, zonas urbanas, roca, vegetación seca |
| 0.2 – 0.4 | Vegetación escasa: pastos, matorral, cultivos en desarrollo |
| 0.4 – 0.6 | Vegetación moderada |
| > 0.6 | Vegetación densa y vigorosa |

**Contexto del Corredor Seco:** enero–marzo es estación seca. Buena parte del bosque seco de Zacapa es caducifolio (pierde las hojas), por lo que en este período se esperan valores de NDVI bajos aun en zonas con cobertura forestal. Los valores altos suelen concentrarse en bosques de mayor altitud, vegetación riparia y cultivos bajo riego.

> Estos rangos son orientativos: la interpretación depende del tipo de cobertura, la época del año y el objetivo del análisis.

## 10. Preparar la carpeta de salida en el proyecto de ArcGIS Pro

El GeoTIFF se guarda en una subcarpeta `NDVI_export` dentro de la carpeta del proyecto abierto. Si la capa ya está en el mapa (por una ejecución anterior), se retira primero para liberar el archivo.

In [ ]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.activeMap

carpeta_salida = os.path.join(aprx.homeFolder, "NDVI_export")
os.makedirs(carpeta_salida, exist_ok=True)

nombre_base = f"NDVI_{DEPTO}_{FECHA_INICIO[:7]}_{FECHA_FIN[:7]}".replace(' ', '_')
ruta_ndvi = os.path.join(carpeta_salida, f"{nombre_base}.tif")

# Retirar del mapa una versión anterior del mismo archivo (evita bloqueos)
if mapa is not None:
    for capa in mapa.listLayers():
        if capa.supports("DATASOURCE") and os.path.normcase(capa.dataSource) == os.path.normcase(ruta_ndvi):
            mapa.removeLayer(capa)
            print(f"  Capa anterior retirada: {capa.name}")

print("Archivo de salida:")
print(f"  {ruta_ndvi}")

## 11. Exportar el NDVI como GeoTIFF

Se descargan los **valores** del NDVI (no la imagen coloreada), reproyectados a `CRS_SALIDA`. Los píxeles fuera del departamento reciben el valor `NODATA` y luego se declaran como *NoData* para que ArcGIS Pro los ignore.

> ℹ️ `ee_export_image` descarga directamente y tiene un límite de tamaño por solicitud (del orden de decenas de MB). Para áreas grandes o 10 m de resolución use `geemap.ee_export_image_to_drive()` y descargue desde Google Drive.

In [ ]:
geemap.ee_export_image(
    ndvi.toFloat(),
    filename=ruta_ndvi,
    scale=ESCALA_M,
    crs=CRS_SALIDA,
    region=geom,
    unmask_value=NODATA,
    file_per_band=False
)

if not os.path.exists(ruta_ndvi):
    raise RuntimeError("La exportación no generó el archivo. Revise el tamaño del área o la resolución.")

arcpy.management.SetRasterProperties(ruta_ndvi, nodata=f"1 {NODATA}")

tam_mb = os.path.getsize(ruta_ndvi) / 1e6
print(f"✓ GeoTIFF exportado ({tam_mb:.1f} MB)")
print(f"  {ruta_ndvi}")

## 12. Agregar el GeoTIFF al mapa activo de ArcGIS Pro

La capa se agrega con un nombre descriptivo y, si está disponible, con la rampa de color *Red-Yellow-Green*.

In [ ]:
if mapa is None:
    print("⚠ No hay un mapa activo. Abra un mapa en ArcGIS Pro y vuelva a ejecutar esta celda.")
else:
    capa_ndvi = mapa.addDataFromPath(ruta_ndvi)
    capa_ndvi.name = f"NDVI {DEPTO} ({FECHA_INICIO} a {FECHA_FIN})"

    # Simbología: estiramiento con rampa Rojo-Amarillo-Verde
    try:
        sym = capa_ndvi.symbology
        sym.updateColorizer('RasterStretchColorizer')
        rampas = aprx.listColorRamps('Red-Yellow-Green*')
        if rampas:
            sym.colorizer.colorRamp = rampas[0]
        capa_ndvi.symbology = sym
        print("✓ Simbología Red-Yellow-Green aplicada")
    except Exception as e:
        print(f"  (Simbología por defecto; ajústela manualmente si lo desea: {e})")

    print(f"✓ Capa agregada al mapa '{mapa.name}': {capa_ndvi.name}")

## ✅ Resumen

**Sentinel-2 → Earth Engine → filtros → máscara de nubes (SCL) → mediana → NDVI → estadísticas → geemap → GeoTIFF → ArcPy → ArcGIS Pro**

Este flujo combina la capacidad de procesamiento en la nube de Google Earth Engine con el entorno de trabajo institucional de ArcGIS Pro: el procesamiento pesado ocurre en los servidores de Google y a su computadora solo llega el resultado final, listo para análisis y cartografía.

---

## 🧪 Retos para practicar

1. **Otro departamento:** cambie `DEPTO` a `'Chiquimula'`, `'El Progreso'` o `'Jutiapa'` y ejecute todo de nuevo.
2. **Estación seca vs. lluviosa:** genere un segundo NDVI para septiembre–octubre y calcule la diferencia (`ndvi_lluviosa.subtract(ndvi_seca)`). ¿Dónde cambia más la vegetación?
3. **Más resolución:** exporte a `ESCALA_M = 10`. ¿Qué pasa con el tamaño del archivo? ¿Cuándo conviene usar `ee_export_image_to_drive()`?
4. **Otro índice:** calcule el **NDMI** (humedad de la vegetación) con `normalizedDifference(['B8', 'B11'])`.

## 📚 Recursos

- Catálogo Sentinel-2 SR Harmonized: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Documentación de geemap: https://geemap.org
- Registro de proyectos en Earth Engine: https://code.earthengine.google.com/register